In [1]:
import geemap
import ee

In [2]:
Map = geemap.Map()
Map

Map(center=[40, -100], controls=(WidgetControl(options=['position'], widget=HBox(children=(ToggleButton(value=…

In [3]:
file = r"E:\Data\GIS\temp.shp"
points = geemap.shp_to_ee(file)
Map.addLayer(points, {}, "p")
Map.centerObject(points, 4)

In [4]:
p1 = ee.Feature(points.first())
r1 = p1.buffer(500000)
t1 = r1.bounds()

In [5]:
Map.addLayer(r1, {}, 'r1')
Map.addLayer(t1, {}, 't1')

In [6]:
def fun(img):
    return img.select("B2", 'B3', 'B4')

In [7]:
image = ee.ImageCollection("COPERNICUS/S2") \
    .filterBounds(t1.geometry())\
    .filterDate('2017', '2021') \
    .map(fun)\
    .filter(ee.Filter.calendarRange(160, 240, 'day_of_year'))\
    .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 5)\
    .sort('CLOUDY_PIXEL_PERCENTAGE', False)

In [ ]:
names = image.aggregate_array("system:id").getInfo()

In [ ]:
names

In [ ]:
i = []
ns = []
for name in names:
    if name[-6:] not in i:
        i.append(name[-6:])
        ns.append(name)
ns

In [ ]:
image.aggregate_array("CLOUDY_PIXEL_PERCENTAGE").getInfo()

In [ ]:
imgs = []
for i in ns:
    imgs.append(ee.Image(i))
images = ee.ImageCollection(imgs).map(fun)

In [ ]:
images.aggregate_array("CLOUDY_PIXEL_PERCENTAGE").getInfo()

In [ ]:
Map.centerObject(img, 12)

In [ ]:
img = ee.Image('COPERNICUS/S2_SR/20200629T031539_20200629T031704_T49SGU')

In [ ]:
Map.addLayer(image, {'bands': ['B4', 'B3', 'B2'], "min": 50, "max": 1500}, "NDVI")

In [ ]:
Map.addLayer(img, {'bands': ['B4', 'B3', 'B2'], "min": 50, "max": 1500}, "N")

In [ ]:
img.geometry().area().getInfo()

In [ ]:
t1.geometry().area().getInfo()

In [ ]:
img.get("CLOUD_COVERAGE_ASSESSMENT").getInfo()

In [ ]:
img = img.select("B[2-4]")

In [ ]:
img.bandNames().getInfo()

In [ ]:
i = image.mosaic()

In [ ]:
file = "E:/Data/temp.tif"
geemap.ee_export_image(image.mosaic(), filename=file, scale=55, region=img.geometry())

In [ ]:
geemap.ee_export_image_to_drive(img.select('B4', 'B3', 'B2'), description='20200826T032541_20200826T032643_T49SCU',
                                folder='Image', scale=10)